# M3

In [ ]:
import os
import sys
from pathlib import Path

for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

ROOT = Path.cwd().resolve()
while not (ROOT / "utils.py").is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Open this notebook inside the code folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from simulation.run_simulation import SimulationConfig, run_simulation
from utils import ratio_table
from IPython.display import HTML


In [ ]:
n = 500  # Number of labeled observations (X and y).
N = 5000  # Number of unlabeled observations (X only).
reps = 100  # Number of repeated experiments for each setting.
dimensions = range(2, 16)  # Numbers of input columns to test, from 2 to 15.
seed = 1  # Random seed for the first repetition.
noise_sigma = 1.0  # Standard deviation of random noise added to the response.
jobs = 12  # Maximum number of repetitions run at the same time.
estimators = [  # Methods to fit to each repeated sample.
    "ata_sbf", "ata_csbf",
    "lp_nw_mode_1", "lp_ll_mode_1",  # Simulation grid: 20 log-spaced values from 0.35*n^(-1/5) to 0.5.
    "krr_m", "krr_r",
    "ease_nw", "ease_ll", "ease_m", "ease_r", "pi",
]

# EASE splits the labeled sample into K groups.
# Fit on K - 1 groups and predict the remaining group.
K = 5
# EASE uses SS-SIR to combine the input columns into r new variables.
# Fit the regression using these r variables.
r = 2
# EASE's SS-SIR splits the response range into H equal-width intervals (slices).
# n // 5 means n divided by 5, rounded down.
H = n // 5
# Split the training data into cv_folds groups to compare tuning candidates.
# Fit on all but one group and measure prediction error on the remaining group.
# For EASE, this split uses only the K - 1 training groups.
cv_folds = 10

experiment_m3 = SimulationConfig(
    model="M3",
    estimators=estimators,
    dimensions=dimensions,
    n=n,
    N=N,
    reps=reps,
    seed=seed,
    noise_sigma=noise_sigma,
    K=K,
    r=r,
    H=H,
    cv_folds=cv_folds,
    jobs=jobs,
    run_name=f"M3_n{n}_N{N}",  # Output folder name for this run.
)

summary_m3, records_m3 = run_simulation(experiment_m3)

In [ ]:
table_m3 = ratio_table(summary_m3, "M3")
shown_m3 = table_m3.map(lambda value: f"{value:.3f}")
shown_m3 = shown_m3.mask(
    table_m3.eq(table_m3.max()), "<b>" + shown_m3 + "</b>"
)
HTML(shown_m3.to_html(escape=False))